#**Projeto Cronos**
##CHALLENGE LOCAWEB 2026
Integrantes:

* Bruno Rosa - RM563779
* Danilo Alves - RM564109
* Enzo Cremaschi - RM562058
* Vinícius Macedo - RM561911

# 02 — Camada Silver | Projeto Cronos (Locaweb Challenge 2026)

Na Camada Silver entram as regras de negócio de limpeza e consistência. Cada decisão abaixo tem a evidência da EDA que a sustenta citada em markdown.

**Transformações desta camada, em ordem:**
1. Exclusão de 2023-2024 (perfil categórico distinto, não é sazonalidade)
2. Deduplicação por `Número`
3. Validação/flag de consistência temporal
4. Tratamento de `Duração` (capping + versão log, sem descartar a coluna bruta)
5. Normalização determinística de texto (`Descrição resumida` -> `descricao_limpa`)
6. Flags de cobertura de categorização (`Produto`/`Categoria`/`Subcategoria`)
7. Profiling comparativo Bronze vs. Silver

**Pré-requisito:** `01_bronze_ingestao.ipynb` já executado e `lw_incidentes_raw.parquet` disponível.


In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import sys
sys.path.append('/content/drive/MyDrive/cronos_project')

import pandas as pd
import numpy as np

from utils import (
    setup_logging,
    PROJECT_PATHS,
    EXPECTED_COLUMNS,
    load_parquet_layer,
    save_parquet_with_metadata,
    profile_dataframe,
    diff_row_count,
    clean_text_ptbr,
    cap_outliers,
)

logger = setup_logging('silver')
PROJECT_PATHS.ensure_dirs()

In [15]:
bronze_path = PROJECT_PATHS.bronze / 'lw_incidentes_raw.parquet'
df = load_parquet_layer(bronze_path, logger=logger)
df.head()

2026-08-15 18:39:05 | INFO     | silver | Lido de /content/drive/MyDrive/cronos_project/data/bronze/lw_incidentes_raw.parquet: 122543 linhas x 19 colunas (3 colunas de metadado: ['_ingested_at', '_source_layer', '_source_hash']).


INFO:silver:Lido de /content/drive/MyDrive/cronos_project/data/bronze/lw_incidentes_raw.parquet: 122543 linhas x 19 colunas (3 colunas de metadado: ['_ingested_at', '_source_layer', '_source_hash']).


,Número,Prioridade,Produto,Categoria,Subcategoria,Grupo designado,Item de configuração,Aberto,Resolvido,Encerrado,...,Descrição resumida,Solução,Aberto por,Incidente Pai,Status,Entrou para KPI?,KPI Violado?,_ingested_at,_source_layer,_source_hash
0,INC8654273,3 - Média,None,None,None,Team14,IC00001,2025-12-31 23:45:18,NaT,2025-12-31 23:45:32,...,Problem: Apache Busy Workers,None,Monitoramento,None,Sem Intervenção,NAO,None,2026-08-15T18:22:37.442305+00:00,bronze,87bab6e0625093ad
1,INC8654270,4 - Baixa,None,None,None,Team14,IC00002,2025-12-31 23:39:36,NaT,2025-12-31 23:43:05,...,Problem: Check Application Monitoring,None,Monitoramento,None,Sem Intervenção,NAO,None,2026-08-15T18:22:37.442305+00:00,bronze,87bab6e0625093ad
2,INC8654264,4 - Baixa,None,None,None,Team14,None,2025-12-31 23:23:10,NaT,2025-12-31 23:25:00,...,Problem: Alarm Application Monitoring database...,None,Monitoramento,None,Sem Intervenção,NAO,None,2026-08-15T18:22:37.442305+00:00,bronze,87bab6e0625093ad
3,INC8654263,4 - Baixa,None,None,None,Team14,None,2025-12-31 23:23:07,NaT,2025-12-31 23:24:57,...,Problem: Alarm Application Monitoring coupons ...,None,Monitoramento,None,Sem Intervenção,NAO,None,2026-08-15T18:22:37.442305+00:00,bronze,87bab6e0625093ad
4,INC8654262,4 - Baixa,None,None,None,Team14,IC00003,2025-12-31 23:23:05,NaT,2025-12-31 23:23:47,...,Problem: Check Application Monitoring,None,Monitoramento,None,Sem Intervenção,NAO,None,2026-08-15T18:22:37.442305+00:00,bronze,87bab6e0625093ad


## 1. Exclusão de 2023-2024

**Evidência:** 2023-2024 somam 732 registros (0,60% da base). O perfil categórico desses registros é sistematicamente diferente de 2025 — 73,6% `Manual` (vs. 14,5% em 2025), 89,6% `Encerrado Automaticamente` (vs. 21,5%), predominância de Prioridade Média (64,5% vs. 33,9%). Isso não é "sazonalidade" ou "mudança de regime gradual" — é mais consistente com um período de teste/pré-lançamento do sistema de tickets (teoria). Manter esses registros distorceria qualquer análise agregada (Desafio 1 e Desafio 2), não só o treino do SARIMAX.

**Decisão:** excluir 2023-2024 de toda a base de trabalho a partir daqui, não apenas do recorte de treino do SARIMAX.

In [4]:
df_antes_filtro_ano = df.copy()
df = df[df['Aberto'].dt.year == 2025].copy()
diff_row_count(df_antes_filtro_ano, df, 'exclusao_2023_2024', logger=logger)

2026-08-15 18:36:28 | INFO     | silver | [exclusao_2023_2024] 122543 -> 121811 linhas (-732, -0.60%).


INFO:silver:[exclusao_2023_2024] 122543 -> 121811 linhas (-732, -0.60%).


## 2. Deduplicação por `Número`

**Evidência (EDA completa, Seção 4):** nenhuma duplicata de linha inteira nem de `Número` foi encontrada na base original — mas essa checagem é repetida aqui, na Silver, e não só assumida da EDA, porque a exclusão de 2023-2024 acima poderia teoricamente introduzir um cenário diferente.

In [5]:
n_duplicatas_numero = df['Número'].duplicated().sum()
n_duplicatas_linha = df.duplicated().sum()
logger.info('Duplicatas de Número: %d | Duplicatas de linha inteira: %d', n_duplicatas_numero, n_duplicatas_linha)

df_antes_dedup = df.copy()
df = df.drop_duplicates(subset=['Número'], keep='first')
diff_row_count(df_antes_dedup, df, 'dedup_numero', logger=logger)

2026-08-15 18:36:28 | INFO     | silver | Duplicatas de Número: 0 | Duplicatas de linha inteira: 0


INFO:silver:Duplicatas de Número: 0 | Duplicatas de linha inteira: 0


2026-08-15 18:36:28 | INFO     | silver | [dedup_numero] 121811 -> 121811 linhas (+0, 0.00%).


INFO:silver:[dedup_numero] 121811 -> 121811 linhas (+0, 0.00%).


## 3. Consistência temporal — validação e flags

**Evidência (EDA completa, Seção 4):** nenhum caso de `Encerrado < Aberto`. **Evidência (EDA complementar, Seção 9):** 32 casos de `Resolvido > Encerrado`, com diferenças de 1,7h a ~22 dias, concentrados em parte num padrão sistêmico (`Team14`, mesma janela de números de incidente — provável reprocessamento em lote).

**Decisão:** não remover esses 32 casos (é uma fração ínfima, 0,03% da base 2025, e `Resolvido` não é usado como feature de qualquer forma — ver nota de vazamento no dicionário de dados). Em vez disso, sinalizar com uma coluna de flag, preservando a linha intacta para auditoria — apagar silenciosamente perderia o rastro de um padrão que pode importar para quem for investigar depois.

In [6]:
df['_flag_encerrado_antes_aberto'] = df['Encerrado'] < df['Aberto']
df['_flag_resolvido_apos_encerrado'] = df['Resolvido'].notna() & (df['Resolvido'] > df['Encerrado'])

logger.info(
    'Flags de inconsistência temporal: encerrado_antes_aberto=%d | resolvido_apos_encerrado=%d',
    df['_flag_encerrado_antes_aberto'].sum(),
    df['_flag_resolvido_apos_encerrado'].sum(),
)

2026-08-15 18:36:28 | INFO     | silver | Flags de inconsistência temporal: encerrado_antes_aberto=0 | resolvido_apos_encerrado=32


INFO:silver:Flags de inconsistência temporal: encerrado_antes_aberto=0 | resolvido_apos_encerrado=32


## 4. Tratamento de `Duração`

**Evidência (EDA completa + complementar):** `Duração` está em segundos (confirmado batendo com `Encerrado - Aberto`), tem cauda extrema (máximo ~1.021 dias) e é estatisticamente uma mistura de processos de fechamento diferentes (Kruskal-Wallis H=50.053, p≈0 por `Status`). Os outliers acima do p99 (>71 dias) são majoritariamente `Encerrado Automaticamente` (89,6%) mas de origem predominantemente Manual (68,9%) — um padrão operacional (incidente manual sem acompanhamento, fechado por timeout), não erro de sistema.

**Decisão:** manter `Duração` bruta intacta (é evidência, não erro), e adicionar duas colunas derivadas para uso futuro nas camadas seguintes: `duracao_capped_p99` (winsorizada, para qualquer estatística agregada sensível a outliers) e `duracao_log` (log1p, para uso em modelos/visualizações).

**Nota de escopo importante:** o cap em `duracao_capped_p99` é um teto **global** (calculado sobre a base inteira, sem segmentar por `Status`) — é uma winsorização de referência genérica, útil para uma estatística agregada rápida (ex.: "qual a duração típica, sem deixar outliers dominarem a média"). Ela **não substitui** a segmentação por `Status` que o achado do Kruskal-Wallis exige para qualquer comparação justa entre grupos.

In [7]:
df['duracao_capped_p99'] = cap_outliers(df['Duração'], upper_quantile=0.99, logger=logger, label='Duração')
df['duracao_log'] = np.log1p(df['Duração'].clip(lower=0))

df[['Duração', 'duracao_capped_p99', 'duracao_log']].describe()

2026-08-15 18:36:28 | INFO     | silver | cap_outliers [Duração]: teto no p99 = 3627543.50 | 1219 valores afetados (1.00%).


INFO:silver:cap_outliers [Duração]: teto no p99 = 3627543.50 | 1219 valores afetados (1.00%).


,Duração,duracao_capped_p99,duracao_log
count,1.218110e+05,1.218110e+05,121811.000000
mean,1.284931e+05,7.021505e+04,6.928106
std,1.124241e+06,4.166918e+05,2.548712
min,1.000000e+00,1.000000e+00,0.693147
25%,2.620000e+02,2.620000e+02,5.572154
50%,9.570000e+02,9.570000e+02,6.864848
75%,4.123000e+03,4.123000e+03,8.324579
max,2.950547e+07,3.627543e+06,17.200086


## 5. Normalização determinística de texto

**Evidência (EDA completa + complementar):** `Descrição resumida` tem só 14,7% de unicidade, fortemente templatizada (1 template só responde por 23,4% da base), e tem associação moderada-forte com `Prioridade` (Cramér's V 0,423) e fraca-moderada com `Grupo designado` (0,174).

**Decisão:** aplicar limpeza determinística (minúsculas, remoção de acentuação/pontuação, remoção de stopwords em português) sem qualquer aprendizado de vocabulário. A coluna original é preservada; a limpa é adicionada como `descricao_limpa`.

In [8]:
df['descricao_limpa'] = clean_text_ptbr(df['Descrição resumida'])

df[['Descrição resumida', 'descricao_limpa']].head(8)

,Descrição resumida,descricao_limpa
0,Problem: Apache Busy Workers,problem apache busy workers
1,Problem: Check Application Monitoring,problem check application monitoring
2,Problem: Alarm Application Monitoring database...,problem alarm application monitoring database ...
3,Problem: Alarm Application Monitoring coupons ...,problem alarm application monitoring coupons m...
4,Problem: Check Application Monitoring,problem check application monitoring
5,Problem: Alarm Application Monitoring feeds Me...,problem alarm application monitoring feeds mes...
6,Problem: Check Application Monitoring,problem check application monitoring
7,Problem: Disk I/O is overloaded on IC00006,problem disk i is overloaded on ic00006


## 6. Flags de cobertura de categorização

**Evidência (EDA completa, Seção 3.1):** a nulidade de `Produto`/`Categoria`/`Subcategoria` é explicada quase deterministicamente por `Aberto por = Monitoramento` (qui-quadrado 36.727, p≈0) — não é dado perdido, é dado que nunca existiu para chamados automáticos.

**Decisão:** **não imputar** essas colunas (risco de circularidade e de mascarar um sinal potencialmente informativo). Adicionamos uma flag explícita de cobertura, útil para o Desafio 2 documentar com transparência qual fração da base cada conclusão de tendência realmente cobre.

In [9]:
df['flag_categorizado'] = df['Produto'].notna()
logger.info(
    'Cobertura de categorização (2025, pós-limpeza): %.1f%% dos chamados têm Produto preenchido.',
    df['flag_categorizado'].mean() * 100,
)

2026-08-15 18:36:29 | INFO     | silver | Cobertura de categorização (2025, pós-limpeza): 36.0% dos chamados têm Produto preenchido.


INFO:silver:Cobertura de categorização (2025, pós-limpeza): 36.0% dos chamados têm Produto preenchido.


## 7. Profiling comparativo — Bronze vs. Silver

In [10]:
resumo_silver = profile_dataframe(df)
resumo_silver

,dtype,n_nao_nulos,n_nulos,pct_nulos,n_unicos
Incidente Pai,object,15063,106748,87.63,3305
Solução,object,15200,106611,87.52,2
KPI Violado?,object,25156,96655,79.35,2
Resolvido,datetime64[ns],39579,82232,67.51,36874
Código de fechamento,object,40086,81725,67.09,17
Produto,object,43883,77928,63.97,51
Subcategoria,object,44091,77720,63.80,446
Categoria,object,44090,77721,63.80,140
Item de configuração,object,120035,1776,1.46,9098
Aberto,datetime64[ns],121811,0,0.00,121263


In [11]:
logger.info('Shape final da Silver: %d linhas x %d colunas (Bronze tinha %d linhas).', *df.shape, df_antes_filtro_ano.shape[0])

2026-08-15 18:36:30 | INFO     | silver | Shape final da Silver: 121811 linhas x 28 colunas (Bronze tinha 122543 linhas).


INFO:silver:Shape final da Silver: 121811 linhas x 28 colunas (Bronze tinha 122543 linhas).


## 8. Gravação

In [12]:
silver_path = PROJECT_PATHS.silver / 'lw_incidentes_clean.parquet'
save_parquet_with_metadata(df, silver_path, layer='silver', logger=logger)

2026-08-15 18:36:31 | INFO     | silver | Gravado: /content/drive/MyDrive/cronos_project/data/silver/lw_incidentes_clean.parquet (121811 linhas, 6.2 MB).


INFO:silver:Gravado: /content/drive/MyDrive/cronos_project/data/silver/lw_incidentes_clean.parquet (121811 linhas, 6.2 MB).


In [13]:
# Checagem de sanidade pós-gravação
df_checagem = pd.read_parquet(silver_path)
assert df_checagem.shape[0] == df.shape[0], 'Divergência de linhas entre o gravado e o lido de volta!'
assert df_checagem['Aberto'].dt.year.nunique() == 1, 'Ainda há mais de um ano na base — a exclusão de 2023-2024 falhou!'
logger.info('Checagem pós-gravação OK: %d linhas, %d colunas, cobrindo só %s.',
            *df_checagem.shape, df_checagem["Aberto"].dt.year.unique())

2026-08-15 18:36:32 | INFO     | silver | Checagem pós-gravação OK: 121811 linhas, 28 colunas, cobrindo só [2025].


INFO:silver:Checagem pós-gravação OK: 121811 linhas, 28 colunas, cobrindo só [2025].


## 9. Resumo de execução

In [14]:
resumo_execucao = f'''
EXECUÇÃO DA CAMADA SILVER — {pd.Timestamp.now(tz="UTC").isoformat()}
Entrada (Bronze): {df_antes_filtro_ano.shape[0]:,} linhas
Saída (Silver): {df.shape[0]:,} linhas
Transformações aplicadas:
  - Exclusão de 2023-2024 (perfil categórico distinto)
  - Deduplicação por Número ({n_duplicatas_numero} duplicata(s) encontrada(s))
  - Flags de consistência temporal (resolvido_apos_encerrado: {df["_flag_resolvido_apos_encerrado"].sum()} casos)
  - Duração: colunas derivadas duracao_capped_p99 e duracao_log adicionadas (bruta preservada)
  - Texto: descricao_limpa adicionada (bruta preservada, sem vocabulário aprendido)
  - flag_categorizado adicionada ({df["flag_categorizado"].mean()*100:.1f}% de cobertura)
Saída: {silver_path}
'''.strip()

print(resumo_execucao)

EXECUÇÃO DA CAMADA SILVER — 2026-08-15T18:36:32.325516+00:00
Entrada (Bronze): 122,543 linhas
Saída (Silver): 121,811 linhas
Transformações aplicadas:
  - Exclusão de 2023-2024 (perfil categórico distinto)
  - Deduplicação por Número (0 duplicata(s) encontrada(s))
  - Flags de consistência temporal (resolvido_apos_encerrado: 32 casos)
  - Duração: colunas derivadas duracao_capped_p99 e duracao_log adicionadas (bruta preservada)
  - Texto: descricao_limpa adicionada (bruta preservada, sem vocabulário aprendido)
  - flag_categorizado adicionada (36.0% de cobertura)
Saída: /content/drive/MyDrive/cronos_project/data/silver/lw_incidentes_clean.parquet
